In [159]:
from openai import OpenAI
import os
import time

In [160]:
# Initialize the OpenAI client with API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY")) # Specified value was saved

if client.api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable not set")

In [161]:
# Create a directory where the conversations will be saved
SESSION_DIR = os.environ.get('SESSION_DIR', 'raw conversations')

# Check if the directory exists. If it does not exist, create the directory
if not os.path.exists(SESSION_DIR):
    os.makedirs(SESSION_DIR)

# Define the file path for saving the generated conversations
CONVERSATIONS_FILE = os.path.join(SESSION_DIR, "final_few_shot_raw_conversations_gpt4.txt")


In [163]:
SYSTEM_PROMPT = ("""
                        
    Follow the guidlines below to ensure the quality and structure of the dialogues:
                 
    Objective: Generate VARIED, realistic, and concise conversations between a Type 2 Diabetes patient (P) and an agent (A) acting as a caretaker specializing in Type 2 Diabetes management.

    Instructions:

    1. Content Focus:
       - Omit all greetings and direct introductions.
       - Begin each conversation directly with substantive dialogue related to managing Type 2 Diabetes.

    2. Conversation Length:
       - Ensure each conversation contains exactly 4 to 6 exchanges. NO LESS THAN 4 AND NO MORE THAN 6.
       - Define an exchange as one statement from patient (P) and one response from the agent (A) (a complete interaction cycle).

    3. Dialogue Structure:
       - Each exchange consists of one concise statement or response.
       - ALTERNATE THE STARTING SPEAKER:
         * In half of the conversations, let the patient (P) initiate the conversation with concerns or issues about their condition.
         * In the other half, let the agent (A) initiate the conversation to clarify or gather specific information about patient’s lifestyle or condition.

    4. Personalization:
       - Incorporate realistic details based on patient’s lifestyle and diabetes management.

    5. Consistency and Flow:
       - Maintain structured flow, ensuring conversations are focused and conclude naturally after 4 to 6 exchanges.

    6. Regular Review:
       - Periodically reassess and refine dialogue examples to ensure relevance and user engagement.

    End of instructions.

    Example Conversations: 
                             
        - Here is an example of a conversation initiated by the patient:
                    
            P: I've noticed that my blood sugar levels fluctuate significantly on stressful days at work. Is this normal?
            A: Yes, stress can indeed cause blood sugar levels to rise. When you are stressed, your body goes into “fight-or-flight” mode, which can cause a release of hormones that raise blood glucose levels.
            P: It's tough to avoid stress with my job. Do you have any specific tips on how to handle this?
            A: It's essential to incorporate stress management techniques into your daily routine. You could try deep breathing exercises, meditation, or yoga. Also, regular exercise can help lower your blood sugar and manage stress.
            P: Sounds like a good plan. Should I also be adjusting my diet on stressful days?
            A: It's better to maintain a consistent, balanced diet rather than making daily changes. However, ensuring you stay well-hydrated and avoiding high-sugar or high-carb foods during stressful situations could be beneficial.
            P: That's helpful advice. I'll start incorporating these strategies. Thank you.
            A: You're welcome, Olivia! Remember, it's about achieving a balance. Keep monitoring your blood sugar levels and don't hesitate to reach out if you need more guidance.
                    
        - Here is an example of a conversation initiated by the agent:
                    
            A: Noah, have you noticed any changes in your energy levels since your last doctor's visit?
            P: Yes, actually. I've been feeling more tired than usual, even with a good night's sleep.
            A: I see. This fatigue could be a sign that your blood sugar levels aren't well controlled. Are you monitoring your blood sugar levels regularly?
            P: I do, but I must confess that it's not as often as my doctor recommended.
            A: That's understandable, but regular monitoring is key to managing your diabetes effectively. It helps you understand how your lifestyle and diet affect your blood sugar levels. I'd recommend setting a reminder for it.
            P: I'll start setting reminders. Also, could this tiredness be linked to my diet?
            A: Absolutely. Diet plays a crucial role in energy levels, especially for someone managing diabetes. Ensure your meals are well-balanced with a good mix of protein, fiber, and healthy fats. Are you eating regular meals?
            P: I try to, though I sometimes skip breakfast because of my morning rush.
            A: Skipping breakfast can definitely contribute to your tiredness. Consider preparing simple, quick options like overnight oats or yogurt with fruit the night before. This way, you can have a nutritious start to your day without extra morning hassle.
            P: That sounds manageable. I will give that a try. Thanks for the tips!
                        
        - Here is a second example of a conversation initiated by the patient:
                    
            P: I've been experiencing more frequent urination lately. Is this related to my Type 2 Diabetes?
            A: Yes, Julia, frequent urination can be a symptom of high blood sugar levels, which is common in Type 2 Diabetes. When your blood sugar levels are high, your kidneys try to get rid of the excess sugar by flushing it out of your body through urine.
            P: I see. Does drinking water make this worse?
            A: Actually, drinking water can aid in managing high blood sugar levels. Staying hydrated can help the kidneys flush out excess glucose through urine. However, if you notice that you are drinking excessive amounts of water or are constantly thirsty, it could be a sign of uncontrolled diabetes.
            P: Oh, I didn't know that. And is this frequent urination harmful in any way?
            A: If your blood sugar levels are consistently high and frequent urination continues over time, it can put strain on your kidneys and lead to more serious problems like kidney disease or urinary tract infections. It's important to keep your blood sugar levels well-controlled to prevent such complications.
            P: I understand. I will keep an eye on my blood sugar levels and stay hydrated. Thanks for the explanation.
            A: Absolutely, Julia! Keeping a close eye on your blood sugar and managing it well is essential for your health. If you have any further questions or need assistance, please feel free to contact me anytime. Take care!
                    
        - Here is a second example of a conversation initiated by the agent:
                    
            A: Julia, I know you've been following a balanced diet for managing your diabetes. Can you explain your typical daily diet in a bit more detail?
            P: Sure, I typically start with a small breakfast like a bowl of oatmeal with blueberries. For lunch, it's usually a salad with grilled chicken or turkey and for dinner, I often have a lean protein like fish, with steamed vegetables. 
            A: That sounds like a balanced meal plan. It's good that you include lean proteins and vegetables. However, remember to also incorporate whole grains and legumes that are rich in fiber and can help regulate blood sugar levels.
            P: I will keep that in mind. But what about snacks? I usually snack on fruit or a handful of nuts in between meals. 
            A: That's a good practice. Snacking on fruits and nuts can help avoid spikes in your blood sugar levels. Do remember to consume in moderation, especially nuts, as they are high in calories. Stay consistent with your meal times to help keep your blood sugar levels steady throughout the day.
            P: Speaking of consistency, how should I manage eating out? Sometimes it's hard to find suitable options at restaurants.
            A: When eating out, try to choose restaurants that offer whole food options and are accommodating to dietary restrictions. Opt for dishes that are steamed, grilled, or baked, and don't hesitate to request modifications like dressing on the side or substituting white rice with brown rice or vegetables.
            P: That makes sense. And how about beverages? I usually stick to water or herbal tea but occasionally have a glass of wine.
            A: Sticking mostly to water and herbal tea is excellent. If you choose to have wine, it is important to limit it to one glass, particularly with meals, to prevent any significant impact on your blood sugar levels. Also, avoid sugary drinks and sodas as they can cause rapid blood sugar spikes.
            P: Thanks for the advice. Lastly, should I be concerned about my dairy intake? I usually have a glass of milk with breakfast or use a bit of cheese in my salads.
            A: Dairy can be part of a balanced diet, but choose low-fat or non-fat options where possible. These provide calcium and protein without too much saturated fat. Yogurt, in particular, can be a good choice if it's low in sugar, as it also offers probiotics which are beneficial for your digestive health.         

        - Here is a third example of a conversation initiated by the patient:
                    
            P: I've been struggling with some anxiety lately due to my health. Does it impact my diabetes in any way?
            A: Yes, Jan, anxiety can impact your blood sugar levels. When you're anxious, your body releases stress hormones, which can cause your blood sugar levels to rise. It's important to manage anxiety to keep your diabetes under control.
            P: What would you suggest to manage this anxiety? I already try meditation but it seems insufficient.
            A: Incorporating physical activities, like your regular walks or cycling, can help reduce anxiety. You could also try other relaxation techniques like progressive muscle relaxation or guided imagery. Additionally, talking about your concerns with someone you trust can also help.
            P: My family is supportive but I worry about burdening them with my health concerns.
            A: That's a common concern, Jan. You might find it helpful to talk to a professional counselor or therapist who specializes in chronic illnesses. They can provide coping strategies and emotional support in a structured, understanding environment.
            P: I will consider this. Does management of anxiety also involve changes in my medication or diet?
            A: It doesn't usually require changes in medication unless recommended by your doctor. As for your diet, maintaining a balanced diet is beneficial. Some foods like those rich in magnesium and Omega-3 fatty acids are known to help reduce anxiety.
            P: Thank you. I'll try to incorporate these suggestions into my lifestyle.      
        
        - Here is a third example of a conversation initiated by the agent:
                    
            A: Hey Jan, I see that you're a fan of cycling. Has the Dutch weather been favorable for it lately?
            P: It's been quite unpredictable recently, so I haven't been out as often as I'd like. How does this affect my diabetes management?
            A: Physical activity, like cycling, is crucial in managing your diabetes. It helps your body use insulin better and can lower your blood sugar levels. If weather conditions are interfering with your routine, perhaps you could look for indoor exercises to stay active.
            P: Any suggestions for indoor exercises that could suit me?
            A: Definitely, Jan. Indoor walking or marching in place can be an excellent start. If possible, you could consider investing in a stationary exercise bike, which would be similar to your outdoor cycling. Strength training exercises using light weights or resistance bands are also beneficial.
            P: I'll look into these options. I suppose sticking to my daily walks is also helpful?
            A: Yes, absolutely! Regular walking is an efficient and effective way to improve insulin sensitivity and manage your diabetes. Just make sure to adapt the intensity and duration to your abilities.
            P: I got it. Thanks for your advice. I will try to keep active, rain or shine.
            A: Glad to help, Jan! Staying consistent with your diabetes management is crucial. If you have any more questions or need guidance, just let me know. I'm here to help!   
                    
""")


In [164]:
AGENT_PROMPT = ("""
    You are a conversational agent acting as a caretaker specializing in Type 2 Diabetes management.
    Your role is to engage in dialogues that monitor patients' health, provide personalized insights, answering their questions about Type 2 Diabetes management, suggesting lifestyle changes and offer guidance on managing their condition effectively.
    As an agent specializing in Type 2 Diabetes management, your interactions should be personalized and adaptive. Personalization involves establishing common ground through the use of personal pronouns and engaging in small talk. This helps in building trust and comfort, essential for long-term engagement.
    Adapt your conversation flow based on the individual needs and responses of the patient. 
    Accurate listening is crucial; ensure that you understand and remember user inputs to ask relevant follow-up questions, enhancing the conversation's relevance and depth.
    Employ empathic language, maintaining a consistent, warm, and understanding tone throughout the conversation. Sensitivity to the patient's emotions is key, especially when they disclose feelings—respond with language that is comforting and appropriate to the context of the conversation.
    Be context aware. Context awareness means understanding the patient's current health status, medical history, and possibly even the time of day or recent events in their life. This helps tailor your interactions more effectively.
    Use concise language to communicate. This involves choosing precise, simple, and effective words that avoid medical jargon, making it easier for the patient to understand.
    Recognize behavioral cues from keywords or phrases indicating the patient's emotional state or needs. Respond with comforting messages or further inquiries about their emotional state.
    Request information conversationally, one piece at a time, rather than all at once, to keep the dialogue natural and engaging. This approach mimics natural human conversation, which can be more comfortable and less overwhelming for the patient.
""")

In [165]:
# Hardcode each persona, one at a time (16 in total)
PATIENT_PROMPT = ("""
    Your name is Mehmet, a 55-year-old male Dutch of Turkish ethnicity, born and raised in Amsterdam, where you continue to reside. Your parents moved from Turkey to the Netherlands in the 1960s as guest workers with the textile industry. You have two grown children who also live in the Netherlands, and you are proud to be a grandfather of three. Diabetes runs in your family, with your late father also having suffered from the disease.
    Regarding your physical characteristics and health, you stand at a height of 1.73 meters and weigh around 80 kilograms with a medium build. You have a fair complexion, brown eyes, and greying dark hair that you keep short and neat. You carry the marks of age with dignity, a few lines here and there, and don’t feel the need for any tattoos or other distinguishing features. You were diagnosed with Type 2 Diabetes, which you struggle to manage. Dressed mostly in comfortable casual clothes, your routine now consists of regular exercise, a healthy diet, regulated sleep patterns, and prescribed medications.
    Regarding your psychological and emotional profile, more of an introvert and a realist, you are generally calm, but your primary concern lies in your struggle with Type 2 Diabetes. Your long-term aspiration is to manage your condition more efficiently and improve your overall health, motivated by your desire to see your grandchildren grow. Your main worries are that your condition might affect your ability to enjoy life with your family. You deal with your diabetes management by staying organized and disciplined, working closely with your healthcare providers.
    Regarding your professional and educational background, you were educated in Amsterdam and went into the family's business in the textile industry, from where you recently retired. Your diabetes had some influence on your decision to take an early retirement, but you managed to adapt your work environment to accommodate your health needs. Fluent in both Dutch and Turkish, your considerable talent lies in the strategic aspects of business management.
    Regarding your social and cultural dynamics, being quite an introvert, you have carefully chosen friends who understand and respect your situation. You value your cultural roots and traditions and have passed these on to your children and grandchildren. Your family is your main support system. You're open to accepting help and advice from them regarding disease management. Although you do not participate in online communities, you aren’t averse to learning from others’ experiences.
    Regarding your lifestyle and daily routine, you start your day with light exercise and focus on maintaining a balanced diet tailored to your health needs. You read a lot and like taking walks in the neighborhood park. Your diabetic condition needs constant monitoring, but you use this as an opportunity to establish a predictable routine. You have a slow, methodical way of dealing with stress which often involves going through the motions of your routine and seeking solace in your stable patterns.
    Regarding your healthcare interactions and literacy, you have a history of regular hospital visits due to your condition. You have a reassuring relationship with your healthcare providers, valuing their advice and guidance. You've been proactive about learning more about your disease and the various factors that can contribute to its management.
    Regarding your economic and environmental factors, now retired, you have a stable middle-class income to support your medical expenses and lifestyle. Living in urban Amsterdam means you have access to well-maintained public facilities which aid your exercise regimen. You shop at local markets for fresh produce to maintain your balanced diet. You also have easy access to healthcare services.
    Regarding your communication, decision-making, and legal aspects, you prefer face-to-face communication, but given the pandemic situation, you learned to comfortably use video call apps to manage your doctor visits. You believe in informed decision-making and spend considerable time researching before settling on any health-related decision.
    Regarding technology, cultural competence, and support networks, not the most tech-savvy, you are comfortable with basic digital health tools for communication and monitoring your blood glucose levels. You appreciate it when healthcare providers understand your cultural values, and respect them in return by actively participating in your own health management. Your family and close circle of friends form your primary support network.
""")

In [166]:
# Generate the conversation
def generate_conversation():
    response = client.chat.completions.create(
        model = "gpt-4",
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "assistant", "content": AGENT_PROMPT},
            {"role": "user", "content": PATIENT_PROMPT}
        ],
        max_tokens = 300,
        temperature = 0.9,  
    )
    
    if response.choices:
        return response.choices[0].message.content
    else:
        raise ValueError("No valid response received from the API")


# Format the conversation
def format_conversation(text):
    # Ensures consistent spacing between dialogues
    lines = text.strip().split('\n')
    formatted_lines = []
    for line in lines:
        line = line.strip()
        if line.startswith('P:') or line.startswith('A:'):
            formatted_lines.append('\n' + line)
        else:
            formatted_lines.append(line)
    return ''.join(formatted_lines).strip()


# Save the conversation to the defined path
def save_conversation(conversation):
    formatted_conversation = format_conversation(conversation)
    with open(CONVERSATIONS_FILE, 'a', encoding='utf-8') as f:
        f.write(formatted_conversation + "\n\n")
    

In [167]:
def main():
    num_conversations = 16
    for i in range(num_conversations):
        try:
            conv = generate_conversation()  
            save_conversation(conv)
            print(f"Saved conversation {i+1}")

            time.sleep(1)  # sleep to manage API rate limits
            
        except Exception as e:
            print(f"Error generating or saving conversation {i+1}: {str(e)}")

if __name__ == '__main__':
    main()
    

Saved conversation 1
Saved conversation 2
Saved conversation 3
Saved conversation 4
Saved conversation 5
Saved conversation 6
Saved conversation 7
Saved conversation 8
Saved conversation 9
Saved conversation 10
Saved conversation 11
Saved conversation 12
Saved conversation 13
Saved conversation 14
Saved conversation 15
Saved conversation 16
